# DEMONEXT Telescope Sandbox

This notebook is a sandbox for learning how to operate the DEMONEXT telescope.

Unlike the main DEMONEXT sandbox notebook, this one uses the `demonext` classes.

## WARNING: READ THIS

**Never, Ever, Run All in this notebook at once**  

This was designed to test individula telescope operations and document them. Run without care you could
try to drive the telescope places you don't want it to go.

In [2]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, coordinates, and times

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import TETE
from astropy.time import Time

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

## demonext module

For this notebook we need the `Config` and `Telescope` classes.

In [5]:
import demonext
from demonext import config, telescope

## Convenience boolean translations

Dictionaries to translate booleans into human-readable form.

In [8]:
# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}

## Functions



In [11]:
# put something here if we need it

## Load the runtime configuration file

DEMONEXT uses a YAML-formatted runtime configuration file kept in a common directory.

In [14]:
# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# read by instantiating a demonext Config class

try:
    cfg = config.Config(cfgFile)
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")

## Start the runtime logger

DEMONEXT uses the Python logging facility for runtime engineering logs.  The default directory is
obtained from the runtime configuration file loaded above. Engineering logs are have names `engCCYYMMDD.txt`,
where `CCYYMMDD` is the observing day, obtained using the `demonext.obsDate()` method.

In [17]:
logDir = demonext.homePath(cfg.config["directories"]["LogDir"])

logFile = str(Path(logDir) / f"eng{demonext.obsDate()}.txt")

logging.basicConfig(filename=logFile,
                    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
                    filemode="a",
                    level=logging.INFO)

# ID for log entries

logger = logging.getLogger("telNotes")

logger.info("Started the DEMONEXT telescope control development notebook")

## DEMONEXT `Telescope` class

DEMONEXT uses a Sidereal Technology (aka SiTech) ServoController II for telescope mount control.  The SiTech 
controller is operated using the PlaneWave STI (SiTech Interface) app via the `SiTech.Telescope` ASCOM
Telescope class instance.

### `Telescope(cfgFile)` - instantiate a `Telescope` class instance

#### Arguments:

 * `cfgFile` - (string) name of a YAML runtime configuration file (see `Config` class)
 
Create an instance of the `Telescope` class and pass it the name of the runtime configuration file
to use.  If no `cfgFile` is given, it defaults to looking for `demonext.txt` in the user's 
`~\.demonext\config` directory.  Internally it reads the configuration file using a `Config` class instance
to extract the configuration information it needs - particularly observatory site information and any 
site-specific setup instructions. Sensible defaults for telescope control properties are initialized some
of which may be changed later.

The `Telescope()` constructor does not connect to the telescope controller, this is done later with the `connect()` method described below.

#### Methods:

**Connect/disconnect:**
 * `connect()` - connect to the STI telescope control app using ASCOM
 * `disconnect()` - disconnect from the telescope
 
**Telescope position query:**
 * `position()` - read current telescope pointing info
 * `telInfo()` - return a FITS-ready version of `position()` info
 * `ha()` - return the current telescope hour angle
 * `secz()` - return the current telescope secant of zenith distance (approximate airmass)
 
**Telescope mount state queries:**
 * `isHome()` - return True if the telescope mount is at the home position
 * `isParked()` - return True if the telescope is parked
 * `isTracking()` - return True if sidereal tracking on ON, False if OFF
 * `isSlewing()` - return True if the mount is slewing, False if not (see `isTracking()` to test if tracking)
 * `tracking(onoff)` - turn sidereal tracking on/off
 
**Telescope mount moves:**
 * `home()` - tell the telescope to seek the mount mechanical home position (roughly HA=0, Dec=0)
 * `park()` - park the telescope: slew to park positionand turn off sidereal tracking
 * `slewToRADec(appRA,appDec)` - slew the telescope to the given apparent celestial coordinates
 * `slewToAltAz(alt,az)` - slew the telescope to the given alt-az mount position
 
**Coordinate methods:**
 * `precess(RA2000,Dec2000)` - precess J2000 RA/Dec to observation epoch/equinox RA/Dec (apparent RA/Dec)
 * `apparentRADec(RA2000,Dec2000)` - alternative precession method
 * `validRADec(RA,Dec)` - return True if RA, Dec coordinates are within the mount range of motion
 * `validAltAz(alt,az)` - return True if Alt/Az coordinates are within the mount range of motion
 
#### Properties

 * `connected` - (boolean) True if telescope connected the STI app with ASCOM, False if not connected
 * `telName` - (string) name of the telescope returned by the STI app
 * `minAlt` - (float) minimum allowed altitude of the telescope in decimal degrees, set on the STI app
 * `minHA` - (float) minimum hour angle limit, default -11 hours
 * `maxHA` - (float) maximum hour angle limit, default +11 hours
 * `minDec` - (float) minimum declination in decimal degrees given `minAlt` at this site
 
### Example: Create a `Telescope` class instance

Instantiate a `Telescope` class instance using the site general runtime configuration file
to setup the telescope site and other local configuration information.

Once instantiated successfully we can connect to the STI telecope mount controller app (next set of cells below)

In [20]:
# instantiate a telescope instance

try:
    tel = telescope.Telescope(cfgFile)
except Exception as exp:
    msg = f"Cannot get Telescope instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)


### Methods

Telescope command and control is performed using the different methods of the `Telescope` class.

### `connect()` - connect to the telescope mount controller
 
Instantiate a Windows Common Object Module (COM) client ASCOM interface instance and connect
it with the running PlaneWave STI SiTech Interface application. **The STI app must be running
before using `connect()`**.  

The `connected` boolean property is True if the ASCOM connection to the telescope mount controller
is active, False if not.

### `isTracking()` - query if the telescope is tracking

Returns True if the telescope sidereal tracking is on, False if sidereal tracking is off.
See also `tracking()`


### `tracking(onoff)` - turn sidereal tracking on or off

#### Arguments:
 * `onoff` - (string) "on" to start sidereal tracking, "off" to stop sidereal tracking
 

### Example: Connect telescope, report info, and make sure tracking is off.

The code snippet below will connect to the telescope, and if connected query the telescopes 
name with the `telName` property.  

ies.the telescpe Last step after connecting the first time is to make sure the tracking is off.  
We do this by asking the mount controler if the telescope is tracking using the `isTracking()` method, 
then turning it off if it returns True using the `tracking()` method.

In [22]:
# connect to the telescope controller

logger.info("Connecting to telescope")

try:
    tel.connect()
except Exception as exp:
    print(f"Cannot connect to telescope: {exp}")

if tel.connected:
    print(f"Done: Connected to {tel.telName} successfully")
else:
    print(f"Error: Failed to connect to the telescope controller")
    
# turn off tracking if it is on for some reason

if tel.isTracking():
    tel.tracking("off")

Done: Connected to DEMONEXT SiTech Servo Controller at SRO successfully


### `position()` - query current telescope position

#### Arguments:
 * none
 
Query the telescope mount controller and retrieve the current position of the telescope. The mount controller returns the current azimuth, altitude, RA, Dec, and LST.
    
This method computes the hour angle HA using the LST and RA data returned by the mount controller, and compute secZ from the current telescope altitude and HA we computed.

Returns a dictionary with the data (Az, Alt, SecZ, LST, RA, Dec, HA).

See also telInfo(), ha(), and secz()

### `isHome()` - ask if telescope is at the home position

Returns True if telescope is at the mechanical home position, False if not at home.  See also `home()`.

### `isParked()` - ask if the telescope is parked

Returns True if telescope is parked (usually Az=180d, Alt=5d), False if not parked.  See also `park()`.

### `isSlewing()` - ask if the telescope is slewing

Returns True if telescope is slewing to a new position, False if idle.  See also `isTracking()`.


### Example: query position and make a formatted display

Query the mount position and display the current position in local horizon
coordinates (alt/az), celestial coordinates (RA/Dec), and time (LST and HA), plus the secant of the zenith
distance.

Also query the mount status boolean flags (homed, parked, tracking, slewing), and show the current
pointing limits defined by the `minHA`, `maxHA`, `minAlt`, and `minDec` properties.

Print it in a simple human-readable format

In [33]:
# query mount information only, no moves yet

logger.info("querying telescope position")

try:
    telInfo = tel.position()
except Exception as exp:
    print(f"Cannot get telescope position info: {exp}")

print(f"\nTelescope Position:")
print(f"  Alt/Az: {telInfo["Alt"]:.5f} d, {telInfo["Az"]:.5f} d")
print(f"  RA/Dec: {telInfo["RA"]:.5f} h, {telInfo["Dec"]:.5f} d")
print(f"  LST/HA: {telInfo["LST"]:.5f}h, {telInfo["HA"]:.5f} h")
print(f"    SecZ: {telInfo["SecZ"]:.2f}")

# Mount Status

print(f"\nTelescope Mount Status:")
print(f"   At Home? {YesNo[tel.isHome()]}")
print(f"    Parked? {YesNo[tel.isParked()]}")
print(f"  Tracking: {OnOff[tel.isTracking()]}")
print(f"   Slewing? {YesNo[tel.isSlewing()]}")

# Pointing limits

print(f"\nPointing Limits:")
print(f"   HA: {tel.minHA:.1f} to {tel.maxHA:.1f}h")
print(f"  Dec: {tel.minDec:.2f} to 90 d")
print(f"  Alt: {tel.minAlt:.1f} to 90 d")


Telescope Position:
  Alt/Az: -0.00146 d, 180.02382 d
  RA/Dec: 4.69618 h, -52.93115 d
  LST/HA: 4.69875h, 0.00257 h
    SecZ: 99.99

Telescope Mount Status:
   At Home? No
    Parked? Yes
  Tracking: Off
   Slewing? No

Pointing Limits:
   HA: -11.0 to 11.0h
  Dec: -47.93 to 0 d
  Alt: 5.0 to 90 d


### Coordinate functions

Methods implemented in the telescope() class for precessing and converting coordinates. 

We compare the astropy and ASCOM.Astrometry.Transform methods (answer: they differ at about the 5mas level).

Note that `tel.precess()` is agnostic of coordinate format (decimal vs. sexigesimal strings) whereas
`tel.apparentRADec()` **requires** decimal.

In [28]:
RA2000 = 2.0
Dec2000 = 70.0

appRA1, appDec1 = tel.precess(RA2000,Dec2000)
appRA2, appDec2 = tel.apparentRADec(RA2000,Dec2000)

print(f"astropy: {appRA1}h {appDec1}d")
raStr = tel.dec2sex(appRA1,precision=4)
decStr = tel.dec2sex(appDec1,precision=4)
print(f"         {raStr} {decStr}")
print(f"  ASCOM: {appRA2} {appDec2}")
raStr = tel.dec2sex(appRA2,precision=4)
decStr = tel.dec2sex(appDec2,precision=4)
print(f"         {raStr} {decStr}")

dRA = 3600*15*(appRA1 - appRA2) # convert hours to arcsec
dDec = 3600*(appDec1 - appDec2) # convert degrees to arcsec
print(f"  delta: {dRA:.4f} {dDec:.4f} arcsec, astropy vs. ASCOM")


astropy: 2.034818457219845h 70.12986424545868d
         02:02:05.3464 70:07:47.5113
  ASCOM: 2.0348184555832622 70.12986423324543
         02:02:05.3464 70:07:47.5112
  delta: 0.0001 0.0000 arcsec, astropy vs. ASCOM


## Telescope Moves

### IMPORTANT!  READ THIS!

**If running these cells and moving the telescope, make sure you are physically present in the lab,
that the moving volume is clear of obstacles, and you have access to the emergency stop button 
(virtual on the Windows desktop or physical on the telescope electronics box)**

### Home the telescope

Moves the telescope to the home position, which searches for the edge sensors on the HA and Dec
drives, and using fine motion establishes the known mechanical "home" location. This is roughly
at Dec=0, HA=0 (but slightly off because of the exact location of the home sensor setting).

Unlike other telescope motions, homing is a **blocking** (aka synchronous) operation that runs until
it is complete.  Typical time is about 35-40 seconds to home from the parked position.

This is done with the `Telescope` class `home()` method.

In [28]:
print(f"\nHoming the telescope...")    
logger.info(f"Homing the telescope...")

t0 = time.time()
try:
    tel.home()
except Exception as exp:
    print(f"ERROR: {exp}")

if tel.isHome():
    telInfo = tel.position()
    print(f"Done: Telescope at Home: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d,tracking {OnOff[tel.isTracking()]}")
else:
    telInfo = tel.position()
    print(f"Warning: Telescope not at Home: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d, tracking {OnOff[tel.isTracking()]}")

dt = time.time() - t0
print(f"      Execution time: {dt:.2f} seconds")


Homing the telescope...
Done: Telescope at Home: Alt=51.46229d Az=180.14977d,tracking Off
      Execution time: 35.33 seconds


### Park the telescope

Drives the telescope to the parked position and turns off tracing.

Unlike homing, this motion requires watching the slew status or timing out, which is done for you
by the `Telescope` class `park()` method.

Sometimes we've seen the telescope claim it is not parked when it is.  Might be a tolerance parameter
in the STI app on how close is close enough to the nominal park alt/az position.

Typical time to park is about 35 seconds starting from the home position (HA=0,Dec=0)

In [31]:
print(f"\nParking the telescope...")
logger.info("Parking the telescope...")

t0 = time.time()
try:
    tel.park()
except Exception as exp:
    print(f"ERROR: {exp}")
    
if tel.isParked():
    telInfo = tel.position()
    print(f"Done: Telescope is Parked: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d, tracking {OnOff[tel.isTracking()]}")
else:
    telInfo = tel.position()
    print(f"Done: Telescope says not Parked: Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d, tracking {OnOff[tel.isTracking()]}")

dt = time.time() - t0
print(f"      Execution time: {dt:.2f} seconds")


Parking the telescope...
Done: Telescope is Parked: Alt=-0.00142d Az=180.02452d, tracking Off
      Execution time: 32.96 seconds


### Slew the telescope to an Alt/Az position

Slew the telescope to a given altitude and azimuth. We'd use this to "park" by hand instead of using
the internal preset park position, or to point the telescope at a service or calibration position (like
a flat-field screen if we had one).

Like parking, slew motions require watching the slew status or timing out, which is done for you
by the `Telescope` class `slewToAltAz()` method.  The method also hides an oddity of the order of
alt and az parameters in the low-level ASCOM method (reversed because of ASCOM's generic mount model).

Alt/Az position is coarse, and is not as precise as RA/Dec since it is meant for service/calibration
positions not fine positions on the sky.  Accuracy also appears to depend on the drive tracking status,
still trying to work this out.

In [52]:
# requested altitude and azimuth coordinates

reqAlt = 34.5
reqAz = 270.0

if tel.validAltAz(reqAlt,reqAz):
    print("AltAz valid")
else:
    raise ValueError(f"AltAz invalid: {tel.msg}") 

logger.info(f"Slewing telescope in Alt/Az")
    
print(f"\nMoving telescope to Alt={reqAlt:.2f}d Az={reqAz:.2f}d...")

tel.tracking("off")

print(f"  Sidereal tracking is {OnOff[tel.isTracking()]}")

t0 = time.time()
try:
    tel.slewToAltAz(reqAlt,reqAz)
except Exception as exp:
    print(f"ERROR: {exp}")
    
telInfo = tel.position()
dAlt = reqAlt - telInfo['Alt']
dAz = reqAz - telInfo['Az']

# print result, including the offset between actual and target position

print(f"Done: Telescope at Alt={telInfo['Alt']:.5f}d Az={telInfo['Az']:.5f}d, tracking {OnOff[tel.isTracking()]}")
print(f"      offset: dAlt={3600*dAlt:.2f}arcsec dAz={3600*dAz:.2f}arcsec")
dt = time.time() - t0
print(f"      Execution time {dt:.2f} seconds")

AltAz valid

Moving telescope to Alt=34.50d Az=270.00d...
  Sidereal tracking is Off
Done: Telescope at Alt=34.45271d Az=270.03572d, tracking On
      offset: dAlt=170.25arcsec dAz=-128.61arcsec
      Execution time 14.25 seconds


### Slew the telescope to RA/Dec coordinates

Slew the telescope to a given RA and Dec.  

We validate the coordinates  using
the `Telescope` class built-in `validRADec()` method.

Slew motions require watching the slew status or timing out, which is done for you
by the `Telescope` class `slewToRADec()` method.

RA/Dec position is a fine pointing motion, usually within a few arcseconds of the target
relative to the mount pointing mode (on sky? good question).

Below we give the target coordinates in J2000 "catalog" equinox and convert to 
apparent coordinates now using the `apparentRADec()` method before passing them to
`slewToRADec()`.  The calling program can use their own method for converting catalog
to apparent celestial coordinates.

RA is given in decimal hours, Dec in decimal degrees.

Commented out is a prototype method (see above) `precess()` to allow use of sexigesimal and other
formats transparently.

In [36]:
#
# Coordinate input agostic of coordinate format decimal vs. sexigesimal
#

RA2000 = "07:39:18"
Dec2000 = "00:36:59"

appRA, appDec = tel.precess(RA2000,Dec2000)
raStr0 = tel.dec2sex(appRA)
decStr0 = tel.dec2sex(appDec)

# do it

logger.info(f"Slewing telescope in RA/Dec")

# the Telescope class has a built-in RA/Dec validator - if you skip this, slewToRADec() will also run it.

if tel.validRADec(appRA,appDec):
    print(f"RA/Dec valid")
else:
    print(f"RA/Dec invalid: {tel.msg}")
    raise RuntimeException(f"Invalid RA/Dec: {tel.msg}") # good practice, even if slewToRADec() will also gripe
    
# Slew the telescope
   
print(f"Moving telescope to apparent RA/Dec {raStr0} {decStr0}...")

#tel.tracking("on")

t0 = time.time()
try:
    tel.slewToRADec(appRA,appDec)
except Exception as exp:
    print(f"ERROR: {exp}")
dt = time.time() - t0

telInfo = tel.position()

dRA = 3600.0*15*(appRA - telInfo['RA']) # convert to arcsec
dDec = 3600.0*(appDec - telInfo['Dec'])
raStr = tel.dec2sex(telInfo['RA'])
decStr = tel.dec2sex(telInfo['Dec'])

print(f"\nDone: Telescope at {raStr} {decStr}, tracking {OnOff[tel.isTracking()]}")
print(f"      dRA = {dRA:.3f} arcsec dDec={dDec:.3f} arcsec")
print(f"      Execution time: {dt:.2f} seconds")

RA/Dec valid
Moving telescope to apparent RA/Dec 07:40:39.91 00:33:16.24...

Done: Telescope at 07:40:39.91 00:33:16.39, tracking On
      dRA = -0.063 arcsec dDec=-0.147 arcsec
      Execution time: 10.76 seconds


## Park/Stow the telescope

How to gracefully park the telescope before disconnecting it. Powering the telescope off is done
in a different class.

In [46]:
logger.info("Stowing the telescope")

try:
    tel.park()
except Exception as exp:
    print(f"ERROR: {exp}")

# make sure tracking is off

tel.tracking("off")

if tel.isParked():
    telInfo = tel.position()
    print(f"Done: Telescope is Parked: tracking {OnOff[tel.isTracking()]}")
else:
    telInfo = tel.position()
    print(f"Done: Telescope says not Parked: tracking {OnOff[tel.isTracking()]}")

# disconnect from the STI app and close the ASCOM instance

tel.disconnect()

print("Disconnected from the STI app")

Done: Telescope is Parked: tracking Off
